[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/099.%20The%20Electric%20Vehicle%20Routing%20Problem%20%28EVRP%29/P99-Tier-2_executed.ipynb)

---


# Problem 99: The Electric Vehicle Routing Problem (EVRP)

## Tier 2: Modified Savings Algorithm

### Purpose and scope

This notebook constructs routes for electric vehicles using a battery-aware adaptation of the Clarke–Wright savings heuristic. It starts with one depot-to-customer-to-depot route per customer, ranks possible customer-route merges by distance savings, and accepts a merge only when its route is feasible under the battery model. If a merged route is infeasible, the heuristic tries inserting one charging station on a route leg.

The input is a set of node coordinates, node types, vehicle battery specifications, and an energy-consumption rate. The notebook computes Euclidean distances and corresponding travel energy. The output is a set of heuristic routes, their distances and energy requirements, and visual and numerical summaries. The method is a heuristic: it does not guarantee a globally optimal solution.

### Modeling assumptions

- Node coordinates are measured in kilometres, and straight-line Euclidean distance represents travel distance.
- Energy consumption is a constant amount per kilometre, independent of load, speed, terrain, weather, or driving style.
- Each route starts at the depot with its vehicle's initial charge. Upon arrival at a charging station, the vehicle is assumed to recharge immediately to full capacity; charging time and cost are not included in the objective.
- The fleet is homogeneous in this example: vehicles have the same battery capacity and initial charge. Individual routes are assigned to vehicles after construction; the heuristic does not model vehicle-specific restrictions.
- Each customer must appear in exactly one route. Charging stations may be visited as needed and are not customers.
- Customer service times are stored in the input data but do not affect energy or route feasibility in this model.
- A merge may add at most one charging-station visit. More complex charging patterns are outside the scope of this heuristic implementation.

### Required concepts and the savings rule

A route is an ordered list of node IDs. A route's **state of charge** is the energy remaining in its battery as it travels along that list. A route is battery-feasible if every leg can be reached with the available charge; reaching a charging station replenishes the battery under the assumption above.

Initially, customer $i$ has its own route, $0 \to i \to 0$, where node $0$ is the depot. For two distinct customers $i$ and $j$, the Clarke–Wright saving is

$$
s_{ij} = d_{0i} + d_{0j} - d_{ij}.
$$

Here, $d_{ab}$ is the distance from node $a$ to node $b$. The saving estimates how much distance is removed by replacing the two separate depot returns with a direct connection between the customers. Candidate pairs are considered in descending order of $s_{ij}$. A pair is merged only when its customers are at route endpoints, the routes are distinct, and the resulting route passes the battery-feasibility check. This endpoint rule preserves the route structure required by the savings algorithm.

For a route $R=(v_0,v_1,\ldots,v_m)$, energy on leg $(v_{k-1},v_k)$ is $e_{v_{k-1}v_k}=\rho d_{v_{k-1}v_k}$, where $\rho$ is the energy-consumption rate. If $b_k$ is the charge after arriving at node $v_k$, then travel requires $b_{k-1} \geq e_{v_{k-1}v_k}$. At a charging station, the charge is reset to battery capacity $B$; at other nodes, the arriving charge is the previous charge minus the leg's energy. The route is feasible if these conditions hold for every leg. In the code, `check_route_feasibility` performs this sequential check, and `can_merge_routes` evaluates candidate merged routes.

The method's main loop sorts customer-pair savings and examines them in that order, updating the route set after each accepted merge. With $n$ customers, there are $O(n^2)$ candidate pairs; sorting takes $O(n^2 \log n)$ time. Route searches and feasibility checks add work proportional to the route lengths for each candidate. The exact runtime therefore depends on the number and lengths of candidate routes examined.

### Algorithm sequence

1. Create one depot–customer–depot route per customer.
2. Compute savings for customer pairs only and sort them from largest to smallest.
3. For each pair, check whether the customers are compatible route endpoints.
4. Test the merged route against the battery constraints. If it is infeasible, try a single charging-station insertion.
5. Replace the two routes with their merged route when the check succeeds. Stop when the requested number of routes is reached or no further feasible merge is available.

The example contains one depot, four customers, two charging stations, and two electric vehicles. Battery capacity is 100 kWh, and energy consumption is 0.8 kWh per kilometre. The charging rate is recorded as an instance parameter, but charging duration is not part of the route objective or feasibility calculation.

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
import itertools
import time

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
@dataclass
class Node:
    """A depot, customer, or charging station in the network."""

    id: int
    x: float
    y: float
    node_type: str
    service_time: float = 0.0


@dataclass
class Vehicle:
    """An electric vehicle with a finite battery."""

    id: int
    battery_capacity: float
    initial_charge: float


@dataclass
class Route:
    """An ordered sequence of node IDs and its calculated route metrics."""

    nodes: List[int]
    vehicle_id: int
    total_distance: float = 0.0
    energy_required: float = 0.0
    charging_stops: List[int] = field(default_factory=list)


@dataclass
class EVRPInstance:
    """Input data and operating parameters for the EVRP example."""

    nodes: List[Node]
    vehicles: List[Vehicle]
    energy_consumption_rate: float
    charging_rate: float


def create_heuristic_instance() -> EVRPInstance:
    """Create the four-customer instance used in this demonstration."""

    nodes = [
        Node(0, 0, 0, "depot"),
        Node(1, 12, 8, "customer", service_time=0.5),
        Node(2, 15, 15, "customer", service_time=0.5),
        Node(3, 8, 18, "customer", service_time=0.5),
        Node(4, 20, 5, "customer", service_time=0.5),
        Node(5, 6, 10, "charging_station"),
        Node(6, 18, 12, "charging_station"),
    ]

    vehicles = [
        Vehicle(0, battery_capacity=100, initial_charge=100),
        Vehicle(1, battery_capacity=100, initial_charge=100),
    ]

    return EVRPInstance(
        nodes=nodes,
        vehicles=vehicles,
        energy_consumption_rate=0.8,
        charging_rate=50.0,
    )


instance = create_heuristic_instance()
customers = [
    index
    for index, node in enumerate(instance.nodes)
    if node.node_type == "customer"
]
charging_stations = [
    index
    for index, node in enumerate(instance.nodes)
    if node.node_type == "charging_station"
]

print(
    f"Created EVRP instance with {len(instance.nodes)} nodes "
    f"and {len(instance.vehicles)} vehicles"
)
print(f"Customers: {customers}")
print(f"Charging stations: {charging_stations}")

In [ ]:
def calculate_distance_matrix(nodes: List[Node]) -> np.ndarray:
    """Calculate Euclidean distances between every pair of nodes."""

    coordinates = np.array([(node.x, node.y) for node in nodes], dtype=float)
    differences = coordinates[:, np.newaxis, :] - coordinates[np.newaxis, :, :]
    return np.sqrt(np.sum(differences**2, axis=2))


def calculate_energy_matrix(
    distances: np.ndarray, consumption_rate: float
) -> np.ndarray:
    """Convert each distance into travel energy using a constant rate."""

    return distances * consumption_rate


distances = calculate_distance_matrix(instance.nodes)
energy_consumption = calculate_energy_matrix(
    distances, instance.energy_consumption_rate
)

print("Distance matrix (km):")
print(np.round(distances, 2))
print("\nEnergy-consumption matrix (kWh):")
print(np.round(energy_consumption, 2))

### Distance, energy, and battery feasibility

The distance matrix stores the length of every possible leg, and multiplying it by the energy-consumption rate gives the energy matrix. These matrices let the route-building functions evaluate a proposed leg without recomputing geometry.

A route's total energy is not, by itself, enough to establish feasibility when charging stations are present. Feasibility depends on the energy available before each leg. The check below therefore follows the route in order, subtracts each leg's energy, and restores the battery to full capacity upon reaching a charging station. The initial charge is passed explicitly so the check does not rely on a hidden global value.

In [ ]:
def check_route_feasibility(
    route: List[int],
    instance: EVRPInstance,
    distances: np.ndarray,
    energy_matrix: np.ndarray,
    initial_charge: float,
) -> Tuple[bool, float, List[int]]:
    """Check each route leg and reset charge at visited stations.

    Returns feasibility, the charge at the end of the checked route, and
    the charging-station nodes visited along the route.
    """

    if len(route) < 2:
        raise ValueError("A route must contain at least two nodes.")
    if route[0] != 0 or route[-1] != 0:
        raise ValueError("Every route must start and end at depot node 0.")
    if not 0 <= initial_charge <= instance.vehicles[0].battery_capacity:
        raise ValueError("Initial charge must be within the vehicle's capacity.")

    battery_level = initial_charge
    charging_points = []
    tolerance = 1e-9

    for current_node, next_node in zip(route[:-1], route[1:]):
        energy_needed = energy_matrix[current_node, next_node]
        if battery_level + tolerance < energy_needed:
            return False, battery_level, charging_points

        battery_level -= energy_needed
        if instance.nodes[next_node].node_type == "charging_station":
            charging_points.append(next_node)
            battery_level = instance.vehicles[0].battery_capacity

    return True, battery_level, charging_points


def find_nearest_charging_station(
    current_node: int,
    target_node: int,
    instance: EVRPInstance,
    distances: np.ndarray,
) -> Optional[int]:
    """Choose the station with the shortest extra distance for one detour."""

    station_ids = [
        index
        for index, node in enumerate(instance.nodes)
        if node.node_type == "charging_station"
    ]
    if not station_ids:
        return None

    return min(
        station_ids,
        key=lambda station: (
            distances[current_node, station]
            + distances[station, target_node]
            - distances[current_node, target_node]
        ),
    )


test_route = [0, 1, 0]
feasible, final_battery, charging = check_route_feasibility(
    test_route,
    instance,
    distances,
    energy_consumption,
    instance.vehicles[0].initial_charge,
)
print(
    f"Test route {test_route} feasible: {feasible}; "
    f"final battery: {final_battery:.2f} kWh; "
    f"charging stops: {charging}"
)

### Initial routes and customer-pair savings

The savings method begins with separate routes so every customer is already covered. It then estimates the distance benefit of joining customer pairs. Charging stations are excluded from this pair list because they are optional intermediate stops, not delivery customers. The savings formula ranks candidates; battery feasibility and endpoint compatibility determine whether a candidate can actually be merged.

In [ ]:
def calculate_savings_matrix(
    distances: np.ndarray, customer_ids: List[int]
) -> np.ndarray:
    """Calculate Clarke-Wright savings for customer pairs only."""

    depot = 0
    savings = np.zeros_like(distances, dtype=float)

    for i in customer_ids:
        for j in customer_ids:
            if i != j:
                savings[i, j] = (
                    distances[depot, i]
                    + distances[depot, j]
                    - distances[i, j]
                )

    return savings


def initialize_routes(
    instance: EVRPInstance,
    distances: np.ndarray,
    energy_matrix: np.ndarray,
) -> List[Route]:
    """Create one depot-customer-depot route for each customer."""

    routes = []
    for customer in [
        index
        for index, node in enumerate(instance.nodes)
        if node.node_type == "customer"
    ]:
        route_nodes = [0, customer, 0]
        total_distance = sum(
            distances[a, b] for a, b in zip(route_nodes[:-1], route_nodes[1:])
        )
        total_energy = sum(
            energy_matrix[a, b]
            for a, b in zip(route_nodes[:-1], route_nodes[1:])
        )
        routes.append(
            Route(
                nodes=route_nodes,
                vehicle_id=0,
                total_distance=total_distance,
                energy_required=total_energy,
            )
        )

    return routes


savings_matrix = calculate_savings_matrix(distances, customers)
initial_routes = initialize_routes(instance, distances, energy_consumption)

print("Initial routes:")
for route_number, route in enumerate(initial_routes, start=1):
    print(
        f"Route {route_number}: {route.nodes}, "
        f"distance: {route.total_distance:.2f} km, "
        f"energy: {route.energy_required:.2f} kWh"
    )

savings_list = [
    (i, j, savings_matrix[i, j])
    for i in customers
    for j in customers
    if i != j and savings_matrix[i, j] > 0
]
savings_list.sort(key=lambda item: item[2], reverse=True)

print("\nLargest customer-pair savings (km):")
for i, j, saving in savings_list[:10]:
    print(f"Customers {i} and {j}: {saving:.2f}")

### Construct routes with the modified savings algorithm

For each candidate pair, the algorithm locates the routes containing the two customers. A merge is considered only when the customers are at compatible route endpoints. The candidate route is checked leg by leg. If it is infeasible, the algorithm tries inserting the geometrically nearest charging station on one leg at a time; it accepts an insertion only if the complete resulting route passes the same feasibility check.

The charging-station choice is deliberately limited: the nearest detour is tried for a leg, and the implementation adds at most one station to a candidate merged route. Therefore, failure to find a merge does not prove that no feasible route exists. The requested route count is a stopping target, not a guarantee; the algorithm can finish with more routes if no examined merge is accepted.

In [ ]:
def can_merge_routes(
    route1: Route,
    route2: Route,
    merge_type: str,
    instance: EVRPInstance,
    distances: np.ndarray,
    energy_matrix: np.ndarray,
) -> Tuple[bool, Optional[List[int]]]:
    """Build and battery-check a candidate route, with one station insertion."""

    if merge_type == "end-start":
        merged_nodes = route1.nodes[:-1] + route2.nodes[1:]
    elif merge_type == "end-end":
        merged_nodes = route1.nodes[:-1] + route2.nodes[::-1][1:]
    else:
        return False, None

    vehicle = instance.vehicles[route1.vehicle_id]
    feasible, _, _ = check_route_feasibility(
        merged_nodes,
        instance,
        distances,
        energy_matrix,
        vehicle.initial_charge,
    )
    if feasible:
        return True, merged_nodes

    for position in range(len(merged_nodes) - 1):
        current = merged_nodes[position]
        target = merged_nodes[position + 1]
        station = find_nearest_charging_station(
            current, target, instance, distances
        )
        if station is None:
            continue

        route_with_station = (
            merged_nodes[: position + 1]
            + [station]
            + merged_nodes[position + 1 :]
        )
        feasible_with_station, _, _ = check_route_feasibility(
            route_with_station,
            instance,
            distances,
            energy_matrix,
            vehicle.initial_charge,
        )
        if feasible_with_station:
            return True, route_with_station

    return False, None


def modified_savings_algorithm(
    instance: EVRPInstance,
    distances: np.ndarray,
    energy_matrix: np.ndarray,
    max_vehicles: int = 2,
) -> List[Route]:
    """Construct EV routes by considering customer savings in descending order."""

    if max_vehicles < 1:
        raise ValueError("max_vehicles must be at least 1.")

    print("Running the modified savings algorithm...")
    routes = initialize_routes(instance, distances, energy_matrix)
    customer_ids = [
        index
        for index, node in enumerate(instance.nodes)
        if node.node_type == "customer"
    ]
    savings = calculate_savings_matrix(distances, customer_ids)
    candidate_savings = [
        (i, j, savings[i, j])
        for i in customer_ids
        for j in customer_ids
        if i != j and savings[i, j] > 0
    ]
    candidate_savings.sort(key=lambda item: item[2], reverse=True)
    print(f"Starting with {len(routes)} routes")

    merge_count = 0
    for i, j, saving_value in candidate_savings:
        if len(routes) <= max_vehicles:
            break

        route_i = next(
            (route for route in routes if i in route.nodes and j not in route.nodes),
            None,
        )
        route_j = next(
            (route for route in routes if j in route.nodes and i not in route.nodes),
            None,
        )
        if route_i is None or route_j is None or route_i is route_j:
            continue

        merged_nodes = None
        if route_i.nodes[-2] == i and route_j.nodes[1] == j:
            can_merge, candidate_nodes = can_merge_routes(
                route_i,
                route_j,
                "end-start",
                instance,
                distances,
                energy_matrix,
            )
            if can_merge:
                merged_nodes = candidate_nodes

        if (
            merged_nodes is None
            and route_i.nodes[-2] == i
            and route_j.nodes[-2] == j
        ):
            can_merge, candidate_nodes = can_merge_routes(
                route_i,
                route_j,
                "end-end",
                instance,
                distances,
                energy_matrix,
            )
            if can_merge:
                merged_nodes = candidate_nodes

        if merged_nodes is None:
            continue

        total_distance = sum(
            distances[a, b] for a, b in zip(merged_nodes[:-1], merged_nodes[1:])
        )
        total_energy = sum(
            energy_matrix[a, b]
            for a, b in zip(merged_nodes[:-1], merged_nodes[1:])
        )
        merged_route = Route(
            nodes=merged_nodes,
            vehicle_id=route_i.vehicle_id,
            total_distance=total_distance,
            energy_required=total_energy,
            charging_stops=[
                node_id
                for node_id in merged_nodes
                if instance.nodes[node_id].node_type == "charging_station"
            ],
        )

        routes.remove(route_i)
        routes.remove(route_j)
        routes.append(merged_route)
        merge_count += 1
        print(f"Merge {merge_count}: customer-pair saving {saving_value:.2f} km")
        print(f"  New route: {merged_nodes}")
        print(
            f"  Distance: {total_distance:.2f} km; "
            f"energy: {total_energy:.2f} kWh"
        )

    print(f"\nAlgorithm completed. Final routes: {len(routes)}")
    return routes


start_time = time.perf_counter()
final_routes = modified_savings_algorithm(
    instance, distances, energy_consumption, max_vehicles=2
)
computation_time = time.perf_counter() - start_time
print(f"\nComputation time: {computation_time:.6f} seconds")

### Analyze and validate the constructed routes

The analysis reports route-level distance, energy, charging stops, and battery utilization. Battery utilization here is total route energy divided by battery capacity; it is a descriptive ratio, not a substitute for the leg-by-leg feasibility check. A route that recharges can use more total energy than one battery capacity and still be feasible.

The validation checks that every customer appears exactly once, every route starts and ends at the depot, each route is battery-feasible, and the reported route metrics agree with the distance and energy matrices. These checks assess the returned heuristic solution; they do not establish global optimality.

In [ ]:
def analyze_solution(
    routes: List[Route],
    instance: EVRPInstance,
    distances: np.ndarray,
    energy_matrix: np.ndarray,
) -> Dict:
    """Summarize route metrics using the supplied instance and matrices."""

    customer_ids = {
        index
        for index, node in enumerate(instance.nodes)
        if node.node_type == "customer"
    }
    analysis = {
        "total_distance": sum(route.total_distance for route in routes),
        "total_energy": sum(route.energy_required for route in routes),
        "num_routes": len(routes),
        "num_customers": len(customer_ids),
        "route_details": [],
    }

    for route_number, route in enumerate(routes):
        customers_in_route = [
            node_id
            for node_id in route.nodes
            if instance.nodes[node_id].node_type == "customer"
        ]
        charging_in_route = [
            node_id
            for node_id in route.nodes
            if instance.nodes[node_id].node_type == "charging_station"
        ]
        analysis["route_details"].append(
            {
                "route_id": route_number,
                "nodes": route.nodes,
                "customers": customers_in_route,
                "charging_stops": charging_in_route,
                "distance": route.total_distance,
                "energy": route.energy_required,
                "battery_utilization": (
                    route.energy_required
                    / instance.vehicles[route.vehicle_id].battery_capacity
                    * 100
                ),
            }
        )

    return analysis


solution_analysis = analyze_solution(
    final_routes, instance, distances, energy_consumption
)

served_customers = [
    node_id
    for route in final_routes
    for node_id in route.nodes
    if instance.nodes[node_id].node_type == "customer"
]
assert sorted(served_customers) == sorted(customers)

for route in final_routes:
    assert route.nodes[0] == 0 and route.nodes[-1] == 0
    feasible, _, _ = check_route_feasibility(
        route.nodes,
        instance,
        distances,
        energy_consumption,
        instance.vehicles[route.vehicle_id].initial_charge,
    )
    assert feasible
    calculated_distance = sum(
        distances[a, b] for a, b in zip(route.nodes[:-1], route.nodes[1:])
    )
    calculated_energy = sum(
        energy_consumption[a, b]
        for a, b in zip(route.nodes[:-1], route.nodes[1:])
    )
    assert np.isclose(route.total_distance, calculated_distance)
    assert np.isclose(route.energy_required, calculated_energy)

print("Solution analysis")
print("=" * 50)
print(f"Total distance: {solution_analysis['total_distance']:.2f} km")
print(f"Total energy required: {solution_analysis['total_energy']:.2f} kWh")
print(f"Number of routes: {solution_analysis['num_routes']}")
print(f"Customers served: {solution_analysis['num_customers']}")

for detail in solution_analysis["route_details"]:
    path = " -> ".join(f"Node {node_id}" for node_id in detail["nodes"])
    print(f"\nRoute {detail['route_id'] + 1}:")
    print(f"  Path: {path}")
    print(f"  Customers: {detail['customers']}")
    print(f"  Charging stops: {detail['charging_stops']}")
    print(f"  Distance: {detail['distance']:.2f} km")
    print(
        f"  Energy: {detail['energy']:.2f} kWh "
        f"({detail['battery_utilization']:.1f}% of battery capacity)"
    )

### Visualize the solution

The route plot shows the spatial path for each constructed route. The comparison chart reports route distance and energy on separate vertical axes because they have different units. The final panel compares the total distance of the initial singleton routes with the total distance after merging; this distance reduction is computed from the actual route metrics, rather than from a simulated merge history.

In [ ]:
def visualize_heuristic_solution(
    instance: EVRPInstance,
    routes: List[Route],
    analysis: Dict,
    initial_routes: List[Route],
) -> None:
    """Plot route geometry, route metrics, and actual distance reduction."""

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    ax_routes, ax_metrics, ax_distance, ax_summary = axes.flat
    fig.suptitle(
        "Electric Vehicle Routing Problem: Modified Savings Algorithm",
        fontsize=16,
        fontweight="bold",
    )

    node_labels_used = set()
    for index, node in enumerate(instance.nodes):
        if node.node_type == "depot":
            label = "Depot" if "depot" not in node_labels_used else None
            ax_routes.scatter(
                node.x, node.y, s=280, color="black", marker="s", label=label, zorder=5
            )
            ax_routes.annotate(
                "Depot", (node.x, node.y), xytext=(5, 5), textcoords="offset points"
            )
            node_labels_used.add("depot")
        elif node.node_type == "customer":
            label = "Customer" if "customer" not in node_labels_used else None
            ax_routes.scatter(
                node.x, node.y, s=170, color="royalblue", marker="o", label=label, zorder=4
            )
            ax_routes.annotate(
                f"C{index}", (node.x, node.y), xytext=(5, 5), textcoords="offset points"
            )
            node_labels_used.add("customer")
        else:
            label = (
                "Charging station"
                if "charging station" not in node_labels_used
                else None
            )
            ax_routes.scatter(
                node.x, node.y, s=170, color="seagreen", marker="^", label=label, zorder=4
            )
            ax_routes.annotate(
                f"CS{index}", (node.x, node.y), xytext=(5, 5), textcoords="offset points"
            )
            node_labels_used.add("charging station")

    route_colors = plt.cm.tab10(np.linspace(0, 1, max(len(routes), 1)))
    for route_number, route in enumerate(routes):
        x_coordinates = [instance.nodes[node_id].x for node_id in route.nodes]
        y_coordinates = [instance.nodes[node_id].y for node_id in route.nodes]
        ax_routes.plot(
            x_coordinates,
            y_coordinates,
            color=route_colors[route_number],
            linewidth=2,
            alpha=0.8,
            marker="o",
            markersize=4,
            label=f"Route {route_number + 1}",
        )

    ax_routes.set_xlabel("X coordinate (km)")
    ax_routes.set_ylabel("Y coordinate (km)")
    ax_routes.set_title("Constructed routes")
    ax_routes.legend()
    ax_routes.grid(True, alpha=0.3)
    ax_routes.set_aspect("equal", adjustable="datalim")

    route_names = [f"Route {index + 1}" for index in range(len(routes))]
    route_distances = [route.total_distance for route in routes]
    route_energies = [route.energy_required for route in routes]
    positions = np.arange(len(route_names))
    width = 0.35
    ax_metrics.bar(
        positions - width / 2,
        route_distances,
        width,
        label="Distance (km)",
        alpha=0.8,
    )
    energy_axis = ax_metrics.twinx()
    energy_axis.bar(
        positions + width / 2,
        route_energies,
        width,
        label="Energy (kWh)",
        color="darkorange",
        alpha=0.8,
    )
    ax_metrics.set_xlabel("Route")
    ax_metrics.set_ylabel("Distance (km)")
    energy_axis.set_ylabel("Energy (kWh)")
    ax_metrics.set_title("Route distance and energy")
    ax_metrics.set_xticks(positions)
    ax_metrics.set_xticklabels(route_names)
    ax_metrics.legend(loc="upper left")
    energy_axis.legend(loc="upper right")
    ax_metrics.grid(True, axis="y", alpha=0.3)

    initial_distance = sum(route.total_distance for route in initial_routes)
    final_distance = analysis["total_distance"]
    ax_distance.bar(
        ["Initial singleton routes", "Final routes"],
        [initial_distance, final_distance],
        color=["slategray", "teal"],
    )
    ax_distance.set_ylabel("Total distance (km)")
    ax_distance.set_title("Total distance before and after merging")
    ax_distance.grid(True, axis="y", alpha=0.3)

    average_distance = (
        analysis["total_distance"] / analysis["num_routes"]
        if analysis["num_routes"]
        else 0.0
    )
    summary_text = (
        "Modified savings algorithm\n"
        "\n"
        f"Total distance: {analysis['total_distance']:.2f} km\n"
        f"Total energy: {analysis['total_energy']:.2f} kWh\n"
        f"Routes used: {analysis['num_routes']}\n"
        f"Customers served: {analysis['num_customers']}\n"
        f"Average distance per route: {average_distance:.2f} km\n"
        "\n"
        f"Battery capacity: {instance.vehicles[0].battery_capacity} kWh\n"
        f"Energy-consumption rate: {instance.energy_consumption_rate} kWh/km\n"
        f"Charging rate: {instance.charging_rate} kW"
    )
    ax_summary.text(
        0.05,
        0.5,
        summary_text,
        transform=ax_summary.transAxes,
        fontsize=11,
        verticalalignment="center",
        fontfamily="monospace",
        bbox={"boxstyle": "round", "facecolor": "aliceblue", "alpha": 0.9},
    )
    ax_summary.set_title("Solution summary")
    ax_summary.axis("off")

    plt.tight_layout()
    plt.show()


visualize_heuristic_solution(
    instance, final_routes, solution_analysis, initial_routes
)

### Reference comparison

The following reference keeps every customer on its own depot–customer–depot route. It illustrates the distance effect of route merging, but it is not an optimized solution and does not respect the requested two-route target. The comparison is descriptive; it does not establish that the savings heuristic is optimal or that it outperforms an exact method.

In [ ]:
def compare_with_greedy_baseline(
    instance: EVRPInstance,
    distances: np.ndarray,
    energy_matrix: np.ndarray,
    heuristic_routes: List[Route],
) -> Dict:
    """Compare the heuristic with the singleton-route reference."""

    print("Comparing with the singleton-route reference...")
    reference_routes = initialize_routes(instance, distances, energy_matrix)
    reference_analysis = analyze_solution(
        reference_routes, instance, distances, energy_matrix
    )
    heuristic_analysis = analyze_solution(
        heuristic_routes, instance, distances, energy_matrix
    )

    distance_reduction = (
        reference_analysis["total_distance"]
        - heuristic_analysis["total_distance"]
    )
    reduction_percentage = (
        100 * distance_reduction / reference_analysis["total_distance"]
        if reference_analysis["total_distance"]
        else 0.0
    )
    comparison = {
        "distance_reduction": distance_reduction,
        "reduction_percentage": reduction_percentage,
        "route_reduction": (
            reference_analysis["num_routes"] - heuristic_analysis["num_routes"]
        ),
        "heuristic_total": heuristic_analysis["total_distance"],
        "reference_total": reference_analysis["total_distance"],
        "heuristic_routes": heuristic_analysis["num_routes"],
        "reference_routes": reference_analysis["num_routes"],
    }

    print("\nComparison results:")
    print(
        f"Distance reduction: {comparison['distance_reduction']:.2f} km "
        f"({comparison['reduction_percentage']:.1f}%)"
    )
    print(f"Route reduction: {comparison['route_reduction']} routes")
    print(
        f"Heuristic: {comparison['heuristic_total']:.2f} km, "
        f"{comparison['heuristic_routes']} routes"
    )
    print(
        f"Singleton-route reference: {comparison['reference_total']:.2f} km, "
        f"{comparison['reference_routes']} routes"
    )
    return comparison


comparison_results = compare_with_greedy_baseline(
    instance, distances, energy_consumption, final_routes
)

### Scalability experiment

This experiment applies the same modified savings algorithm to instances with different customer counts. Customer coordinates are generated from a fixed seed so that the experiment is reproducible. The charging-station locations, battery parameters, and energy rate follow the example's stated setup. The results describe the instances generated by this code; they are not a general performance guarantee. Runtime measurements can vary with the computing environment.

In [ ]:
def scalability_test() -> List[Dict]:
    """Run the savings heuristic on reproducible instances of varying size."""

    print("Scalability test")
    print("=" * 50)
    test_sizes = [3, 4, 5, 6, 8]
    results = []

    for n_customers in test_sizes:
        np.random.seed(42)
        nodes = [Node(0, 0, 0, "depot")]
        for node_id in range(1, n_customers + 1):
            nodes.append(
                Node(
                    node_id,
                    np.random.uniform(5, 25),
                    np.random.uniform(5, 20),
                    "customer",
                    service_time=0.5,
                )
            )
        nodes.append(Node(n_customers + 1, 8, 12, "charging_station"))
        nodes.append(Node(n_customers + 2, 18, 8, "charging_station"))

        test_instance = EVRPInstance(
            nodes=nodes,
            vehicles=[Vehicle(0, 100, 100), Vehicle(1, 100, 100)],
            energy_consumption_rate=0.8,
            charging_rate=50.0,
        )
        test_distances = calculate_distance_matrix(test_instance.nodes)
        test_energy = calculate_energy_matrix(
            test_distances, test_instance.energy_consumption_rate
        )

        start = time.perf_counter()
        test_routes = modified_savings_algorithm(
            test_instance, test_distances, test_energy, max_vehicles=2
        )
        elapsed = time.perf_counter() - start
        test_analysis = analyze_solution(
            test_routes, test_instance, test_distances, test_energy
        )

        results.append(
            {
                "customers": n_customers,
                "computation_time": elapsed,
                "total_distance": test_analysis["total_distance"],
                "num_routes": test_analysis["num_routes"],
                "total_energy": test_analysis["total_energy"],
            }
        )
        print(
            f"{n_customers} customers: {elapsed:.6f} seconds, "
            f"{test_analysis['total_distance']:.2f} km, "
            f"{test_analysis['num_routes']} routes"
        )

    return results


scalability_results = scalability_test()

### Interpretation and limitations

The modified savings heuristic retains the savings algorithm's central logic: it prioritizes customer pairs by estimated distance reduction and combines routes when the endpoint and battery checks permit. Its battery check adds a sequential energy-feasibility condition, and its charging insertion step provides a limited way to restore feasibility.

The returned routes are feasible under the assumptions implemented here, but the method does not guarantee the minimum total distance. Savings are a ranking rule rather than a proof that a merge belongs to an optimal solution. The one-station insertion limit, the nearest-detour rule, and the endpoint restrictions can prevent the heuristic from finding feasible or shorter routes that would require a different merge order or additional charging stops. Likewise, if the algorithm ends with more routes than requested, that indicates that its examined merges did not reach the target; it does not prove that the target is impossible.

The model also omits charging duration, time windows, vehicle load, battery degradation, traffic, and other operational constraints. These can be incorporated as additional modeling requirements while retaining the savings-based route-construction framework, but they are not evaluated in this example.